<a href="https://colab.research.google.com/github/mlister3/UMR_Momentum/blob/main/MomentumSelectF26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dependencies

In [ ]:
# Import Dependencies
import pandas as pd
import numpy as np
from numpy import object_
import shap
import os
import requests
from google.colab import auth
from google.colab import files
from googleapiclient.discovery import build
import warnings
from io import BytesIO
# Disable all warnings
warnings.filterwarnings("ignore")

# Set Auth to False
g_drive_auth = False

In [ ]:
def auth_user_g_drive():
  print("Initiating Google Drive authentication...")
  try:
    auth.authenticate_user()
    print("Google Drive authenticated successfully.")
    global g_drive_auth
    g_drive_auth = True
  except Exception as e:
    print(f"Authentication failed: {e}")

def request_download_file(fileId):
  service = build('drive', 'v3')
  try:
    # Request the file content
    request = service.files().get_media(fileId=fileId)
    downloaded_file = request.execute()
    file_content = BytesIO(downloaded_file)
    print(f"Successfully downloaded file with ID: {fileId}")
    return file_content
  except Exception as e:
    print(f"Failed to download file with ID {fileId}: {e}")
    print("Please check the file ID and ensure your Google account has access to the file.")

# Authenticate User Google Drive
auth_user_g_drive()

past_cohort_apps_df = pd.read_excel(request_download_file("1Zq2hCmUUcTheJMDn3WTI_lINjOyp-rAj"))
past_past_cohort_apps_df = pd.read_excel(request_download_file("1afWnFNw5jiYF5TDQHh7K9x8JM6ZVrSio"))
current_cohort_apps_df = pd.read_excel(request_download_file("1yCes_7lLUi33yJ9G1uafolMZxa91ru6N"))
current_cohort_coursework_df = pd.read_excel(request_download_file("1pKxPJpFjaeoz3-bvYr_5sXJXWWYr361X"))
past_cohort_help_df = pd.read_excel(request_download_file("1RgKFZQt4-88vBmfwXmsC-Zo1VDvcRhcz"))

## Pre-Processing

In [ ]:
# List of file paths to read
file_paths = [
    '/content/ALEKS Full Class Report F24.xlsx',
    '/content/ALEKS Full Class Report F25.xlsx',
    '/content/ALEKS Full Class Report F26.xlsx'
]

# List to hold individual DataFrames
dfs = []

# Read each Excel file and append to the list
for file in file_paths:
    try:
        df = pd.read_excel(file)
        dfs.append(df)
        print(f"Successfully read {file}")
    except FileNotFoundError:
        print(f"Error: File not found at {file}")
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Concatenate all DataFrames into a single DataFrame if any were read
if dfs:
    ALEKS_df = pd.concat(dfs, ignore_index=True)
    print("Successfully combined all ALEKS reports into ALEKS_df.")

    # Convert specificed columns to datetime type
    datetime_cols = ['placementResults_startDate_0', 'placementResults_startDate_1',
                    'placementResults_startDate_2', 'placementResults_startDate_3', 'placementResults_startDate_4',
                    'placementResults_endDate_0', 'placementResults_endDate_1',
                    'placementResults_endDate_2', 'placementResults_endDate_3', 'placementResults_endDate_4',
                    'knowledgeCheckInfo_startDate_0', 'knowledgeCheckInfo_startDate_1',
                    'knowledgeCheckInfo_startDate_2', 'knowledgeCheckInfo_startDate_3',
                    'knowledgeCheckInfo_startDate_4', 'knowledgeCheckInfo_endDate_0', 'knowledgeCheckInfo_endDate_1',
                    'knowledgeCheckInfo_endDate_2', 'knowledgeCheckInfo_endDate_3',
                    'knowledgeCheckInfo_endDate_4']
    for col in datetime_cols:
      ALEKS_df[col] = pd.to_datetime(ALEKS_df[col])
      # Extract year, month, and day components
      ALEKS_df[col + '_year'] = ALEKS_df[col].dt.year
      ALEKS_df[col + '_month'] = ALEKS_df[col].dt.month
      ALEKS_df[col + '_day'] = ALEKS_df[col].dt.day

      # Drop the original datetime columns after extracting features
    ALEKS_df = ALEKS_df.drop(columns=datetime_cols)

    # Deduplicate based on 'studentInfo_id', keeping the row with the highest 'placementResults_score_0'
    if 'studentInfo_id' in ALEKS_df.columns and 'placementResults_score_0' in ALEKS_df.columns:
        ALEKS_df = ALEKS_df.sort_values(by=['studentInfo_id', 'placementResults_score_0'], ascending=[True, False])
        ALEKS_df = ALEKS_df.drop_duplicates(subset='studentInfo_id', keep='first')

# Update studentInfo_id to 7 diget str
ALEKS_df['studentInfo_id'] = ALEKS_df['studentInfo_id'].astype(str).str.zfill(7)

In [ ]:
hot_all_coursework_df = current_cohort_coursework_df.copy()
hot_all_coursework_df['unique_id'] = hot_all_coursework_df['unique_id'].astype(str).str.zfill(9)
hot_all_coursework_df = hot_all_coursework_df.drop(['datetime_course_created', 'cat_course_external_ref_id', 'num_course_credit', 'cat_course_term_type'], axis=1)

# Initialize the new column 'Course_Sum_work' to 'NONE' by default
hot_all_coursework_df['Course_Sum_work'] = 'NONE'

# Define the mapping rules in a dictionary for easy editing
course_work_math = {
    # MATH
    'ALGEBRA': {'includes': ['ALG'], 'excludes': ['PRE']},
    'GEOMETRY': {'includes': ['GEOM'], 'excludes': []},
    'CALCULUS': {'includes': ['CALC'], 'excludes': ['PRE']},
    'PRECALC': {'includes': ['PRE'], 'excludes': []},
    'STATS': {'includes': ['STAT'], 'excludes': ['CALC']},
    'TRIG': {'includes': ['TRIG'], 'excludes': []},
    'IB MATH SL': {'includes': ['MATH'], 'includes_or': ['IB', 'SL', 'HL'], 'excludes': []},
    'MATH LIBERAL': {'includes': ['MATH'], 'excludes': ['IB', 'STATS', 'SL', 'HL', 'CALC', 'STAT']},
    'DATA ANALYSIS': {'includes': ['DATA', 'ANALYSIS'], 'excludes': []}
}
course_work_sci = {
    # SCI
    'BIOL': {'includes': ['BIOL'], 'excludes': []},
    'CHEM': {'includes': ['CHEM'], 'excludes': []},
    'PHYSICS': {'includes': ['PHYS'], 'excludes': ['PHYSICAL']},
    'PHYSICAL SCIENCE': {'includes': ['PHYSICAL', 'SCIENCE'], 'excludes': ['PHYSICS']},
    'ANATPHYSIO': {'includes': ['ANAT', 'PHYSIO'], 'excludes': []},
    'ANATOMY': {'includes': ['ANAT'], 'excludes': ['PHYS']},
    'BIOMEDICAL SCIENCE': {'includes': ['BIOMED'], 'excludes': ['TERM']},
    'EARTH SCIENCE': {'includes': ['EARTH', 'SCI'], 'excludes': []},
    'FOOD SCIENCE': {'includes': ['FOOD', 'SCI'], 'excludes': []}
}
course_work_mappings = {**course_work_math, **course_work_sci}

# Apply the rules to create 'Course_Sum_work'
for course_type, conditions in course_work_mappings.items():
    current_includes_condition = pd.Series(True, index=hot_all_coursework_df.index)

    # Handle 'includes' (AND conditions)
    if 'includes' in conditions and conditions['includes']:
        for inc_val in conditions['includes']:
            current_includes_condition = current_includes_condition & hot_all_coursework_df['cat_course_name'].str.contains(inc_val, case=False, na=False)

    # Handle 'includes_or' (OR conditions) for inclusions
    if 'includes_or' in conditions and conditions['includes_or']:
        or_part_condition = pd.Series(False, index=hot_all_coursework_df.index)
        for or_inc_val in conditions['includes_or']:
            or_part_condition = or_part_condition | hot_all_coursework_df['cat_course_name'].str.contains(or_inc_val, case=False, na=False)
        current_includes_condition = current_includes_condition & or_part_condition

    # Handle 'excludes' (AND NOT conditions)
    current_excludes_condition = pd.Series(False, index=hot_all_coursework_df.index)
    if 'excludes' in conditions and conditions['excludes']:
        for exc_val in conditions['excludes']:
            current_excludes_condition = current_excludes_condition | hot_all_coursework_df['cat_course_name'].str.contains(exc_val, case=False, na=False)

    hot_all_coursework_df.loc[current_includes_condition & ~current_excludes_condition, 'Course_Sum_work'] = course_type

# Reorder columns: make 'Course_Sum_work' the column before 'cat_course_name'
if 'Course_Sum_work' in hot_all_coursework_df.columns and 'cat_course_name' in hot_all_coursework_df.columns:
    cols = hot_all_coursework_df.columns.tolist()
    # Remove Course_Sum_work from its current position if it exists
    if 'Course_Sum_work' in cols:
        cols.remove('Course_Sum_work')

    # Find the index of 'cat_course_name'
    cat_course_name_idx = cols.index('cat_course_name')

    # Insert 'Course_Sum_work' at that index
    cols.insert(cat_course_name_idx, 'Course_Sum_work')
    hot_all_coursework_df = hot_all_coursework_df[cols]

hot_all_coursework_df = hot_all_coursework_df[hot_all_coursework_df['Course_Sum_work'] != 'NONE']

# New logic for cat_course_lvl_type
# Dictionary for updating cat_course_lvl_type based on conditions
level_type_mappings = {
    'PSEO': {'includes': ['PSEO'], 'includes_or': [], 'excludes': []},
    'AP': {'includes': [], 'includes_or': ['AP', 'ADVANCED PLACEMENT'], 'excludes': []},
    'IB': {'includes': ['IB'], 'includes_or': [], 'excludes': []},
    'HONORS': {'includes': [], 'includes_or': ['HONORS', 'HON'], 'excludes': []},
    'ADVANCED': {'includes': [], 'includes_or': ['ADVANCED', 'ADV'], 'excludes': ['PLACEMENT']},
    'CIS': {'includes': ['CIS'], 'includes_or': [], 'excludes': []},
    'ACCELLERATED': {'includes': [], 'includes_or': ['ACCELERATED', 'ACCEL'], 'excludes': []},
    'PLTW': {'includes': ['PLTW'], 'includes_or': [], 'excludes': []}
}

for level_type, conditions in level_type_mappings.items():
    # Condition 1: cat_course_lvl_type is blank/NaN or an empty string
    blank_level_condition = hot_all_coursework_df['cat_course_lvl_type'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_lvl_type'].isna()

    current_level_includes_condition = pd.Series(True, index=hot_all_coursework_df.index)
    # Handle 'includes' (AND conditions) for cat_course_name
    if 'includes' in conditions and conditions['includes']:
        for inc_val in conditions['includes']:
            current_level_includes_condition = current_level_includes_condition & hot_all_coursework_df['cat_course_name'].str.contains(inc_val, case=False, na=False)

    # Handle 'includes_or' (OR conditions) for cat_course_name
    if 'includes_or' in conditions and conditions['includes_or']:
        or_part_condition = pd.Series(False, index=hot_all_coursework_df.index)
        for or_inc_val in conditions['includes_or']:
            or_part_condition = or_part_condition | hot_all_coursework_df['cat_course_name'].str.contains(or_inc_val, case=False, na=False)
        current_level_includes_condition = current_level_includes_condition & or_part_condition

    current_level_excludes_condition = pd.Series(False, index=hot_all_coursework_df.index)
    # Handle 'excludes' (AND NOT conditions) for cat_course_name
    if 'excludes' in conditions and conditions['excludes']:
        for exc_val in conditions['excludes']:
            current_level_excludes_condition = current_level_excludes_condition | hot_all_coursework_df['cat_course_name'].str.contains(exc_val, case=False, na=False)

    # Apply the update where blank_level_condition AND (current_level_includes_condition AND NOT current_level_excludes_condition) are met
    hot_all_coursework_df.loc[
        blank_level_condition & current_level_includes_condition & ~current_level_excludes_condition,
        'cat_course_lvl_type'
    ] = level_type

# WORK HERE

# Condition 1: If 'cat_course_subject' is blank and 'cat_course_math_sci' is 'MATH' or 'SCIENCE'
blank_subject_condition = hot_all_coursework_df['cat_course_subject'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_subject'].isna()
math_sci_condition = hot_all_coursework_df['cat_course_math_sci'].isin(['MATH', 'SCIENCE'])
hot_all_coursework_df.loc[blank_subject_condition & math_sci_condition, 'cat_course_subject'] = hot_all_coursework_df['cat_course_math_sci']

# Condition 2: If 'cat_course_math_sci' is blank, infer from 'Course_Sum_work'
blank_math_sci_condition = hot_all_coursework_df['cat_course_math_sci'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_math_sci'].isna()

# Get lists of Course_Sum_work values for math and science
math_sum_works = list(course_work_math.keys())
sci_sum_works = list(course_work_sci.keys())

# Apply 'MATH' if Course_Sum_work is in math_sum_works
hot_all_coursework_df.loc[
    blank_math_sci_condition & hot_all_coursework_df['Course_Sum_work'].isin(math_sum_works),
    'cat_course_subject'
] = 'MATH'

# Apply 'SCIENCE' if Course_Sum_work is in sci_sum_works (only if not already set to 'MATH')
hot_all_coursework_df.loc[
    blank_math_sci_condition & hot_all_coursework_df['Course_Sum_work'].isin(sci_sum_works) &
    (hot_all_coursework_df['cat_course_subject'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_subject'].isna()),
    'cat_course_subject'
] = 'SCIENCE'

hot_all_coursework_df = hot_all_coursework_df.drop(['cat_course_name', 'cat_course_math_sci'], axis=1)

hot_all_coursework_df.head(10)

# Define the output file name, save df to excel, download xlsx
#output_file_name = 'CourseWork_All.xlsx'
#hot_all_coursework_df.to_excel(output_file_name, index=False)
#files.download(output_file_name)

In [ ]:
# Create a sequence number for each course within a unique_id
hot_all_coursework_df['course_instance_num'] = hot_all_coursework_df.groupby('unique_id').cumcount()

# Identify columns that are attributes of a single course instance
course_attributes = [col for col in hot_all_coursework_df.columns if col not in ['unique_id', 'course_instance_num']]

# Melt the DataFrame to bring course attributes into 'attribute' and 'value' columns
df_melted = hot_all_coursework_df.melt(
    id_vars=['unique_id', 'course_instance_num'],
    value_vars=course_attributes,
    var_name='attribute',
    value_name='value'
)

# Create the new column names for pivoting (e.g., 'cat_course_name_1', 'num_course_grade_1')
df_melted['new_column_name'] = df_melted['attribute'] + '_' + df_melted['course_instance_num'].astype(str)

# Pivot the DataFrame to get one row per unique_id with course attributes as new columns
coursework_flattened_df = df_melted.pivot_table(
    index='unique_id',
    columns='new_column_name',
    values='value',
    aggfunc='first'
)

# Reset index to make unique_id a regular column again
coursework_flattened_df = coursework_flattened_df.reset_index()

# Clean up column names (remove the 'new_column_name' level from the columns MultiIndex)
coursework_flattened_df.columns.name = None

print("Shape of the new flattened coursework DataFrame:", coursework_flattened_df.shape)
coursework_flattened_df = coursework_flattened_df.dropna(axis=1, how="all")
print("Shape of the new flattened coursework DataFrame after dropna:", coursework_flattened_df.shape)
print("\nFirst 5 rows of the flattened coursework DataFrame:")
coursework_flattened_df.head()

## Pre-Processing Alt

In [ ]:
# List of files paths to read
file_paths = [
    '/content/UMNRO_CRSE_GR_BYTERM_1243.xlsx',
    '/content/UMNRO_CRSE_GR_BYTERM_1245.xlsx',
    '/content/UMNRO_CRSE_GR_BYTERM_1249.xlsx',
    '/content/UMNRO_CRSE_GR_BYTERM_1253.xlsx',
    '/content/UMNRO_CRSE_GR_BYTERM_1255.xlsx',
    '/content/UMNRO_CRSE_GR_BYTERM_1259.xlsx',
    '/content/UMNRO_CRSE_GR_BYTERM_1263.xlsx'
]

# List to hold individual DataFrames
dfs = []

# Read each Excel file and append to the list
for file in file_paths:
    try:
        df = pd.read_excel(file, header=1)
        dfs.append(df)
        print(f"Successfully read {file}")
    except FileNotFoundError:
        print(f"Error: File not found at {file}")
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Concatenate all DataFrames into a single DataFrame if any were read
if dfs:
    UMR_Courses_df = pd.concat(dfs, ignore_index=True)
    print("Successfully combined all UMR Courses reports into UMR_Courses_df.")

In [ ]:
UMR_Courses_Hot_df = UMR_Courses_df.copy()
UMR_Courses_Hot_df = UMR_Courses_Hot_df.drop(columns=['Campus'])

# Define the list of subjects to keep, ordered alphabetically
science_to_keep = ['ANAT', 'BIOC', 'BIOL', 'BSE', 'CHEM', 'ESCI', 'GCD', 'MICB', 'PHSL', 'PHYS']

math_to_keep = ['MATH', 'STAT']

# Filter the DataFrame to only keep rows with these subjects

subjects_to_keep = math_to_keep + science_to_keep

# Filter the DataFrame to only keep rows with these subjects
UMR_Courses_Hot_df = UMR_Courses_Hot_df[UMR_Courses_Hot_df['Subject'].isin(subjects_to_keep)]

# Define list of grades to keep
grades_to_keep = ['D', 'F']

UMR_Courses_Hot_df = UMR_Courses_Hot_df[UMR_Courses_Hot_df['Grade'].isin(grades_to_keep)]

# Create a new column 'Type' based on the subject lists
conditions = [
    UMR_Courses_Hot_df['Subject'].isin(math_to_keep),
    UMR_Courses_Hot_df['Subject'].isin(science_to_keep)
]
choices = ['MATH', 'SCIENCE']
UMR_Courses_Hot_df['Type'] = np.select(conditions, choices, default=None)

# Optionally, display the unique subjects after filtering to confirm
#print("Unique Subjects after filtering:")
print(UMR_Courses_Hot_df['Descr'].unique())

UMR_Courses_Hot_df.head()

# Option to download as XLSX
#output_xlsx_name = 'simplified_courses.xlsx'
#UMR_Courses_Hot_df.to_excel(output_xlsx_name, index=False)
#files.download(output_xlsx_name)

In [ ]:
# List of file paths to read
file_paths = [
    '/content/ALEKS Full Class Report F24.xlsx',
    '/content/ALEKS Full Class Report F25.xlsx',
    '/content/ALEKS Full Class Report F26.xlsx'
]

# List to hold individual DataFrames
dfs = []

# Read each Excel file and append to the list
for file in file_paths:
    try:
        df = pd.read_excel(file)
        dfs.append(df)
        print(f"Successfully read {file}")
    except FileNotFoundError:
        print(f"Error: File not found at {file}")
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Concatenate all DataFrames into a single DataFrame if any were read
if dfs:
    ALEKS_df = pd.concat(dfs, ignore_index=True)
    print("Successfully combined all ALEKS reports into ALEKS_df.")

    # Define columns to keep and their new names
    columns_to_keep_and_rename = {
        'studentInfo_id': 'studentInfo_id',
        'placementResults_nbPlacementsTaken_0': 'count_placements_taken',
        'placementResults_score_0': 'score_0',
        'placementResults_score_1': 'score_1',
        'placementResults_score_2': 'score_2',
        'placementResults_score_3': 'score_3',
        'placementResults_score_4': 'score_4',
        'progressInfo_performance_percentageMastered_0': 'percentMastered'
    }

    # Filter and rename columns
    ALEKS_df = ALEKS_df[list(columns_to_keep_and_rename.keys())].rename(columns=columns_to_keep_and_rename)

    # Deduplicate based on 'studentInfo_id', keeping the row with the highest 'score_0'
    if 'studentInfo_id' in ALEKS_df.columns and 'score_0' in ALEKS_df.columns:
        ALEKS_df = ALEKS_df.sort_values(by=['studentInfo_id', 'score_0'], ascending=[True, False])
        ALEKS_df = ALEKS_df.drop_duplicates(subset='studentInfo_id', keep='first')

# Update studentInfo_id to 7 diget str
ALEKS_df['studentInfo_id'] = ALEKS_df['studentInfo_id'].astype(str).str.zfill(7)

#WORK HERE

# Find the highest score among score_0 to score_4
score_columns = ['score_0', 'score_1', 'score_2', 'score_3', 'score_4']
ALEKS_df['highest_score'] = ALEKS_df[score_columns].max(axis=1)

# Define conditions for score_range columns
conditions = [
    (ALEKS_df['highest_score'] >= 0.00) & (ALEKS_df['highest_score'] <= 0.29),
    (ALEKS_df['highest_score'] >= 0.30) & (ALEKS_df['highest_score'] <= 0.45),
    (ALEKS_df['highest_score'] >= 0.46) & (ALEKS_df['highest_score'] <= 0.54),
    (ALEKS_df['highest_score'] >= 0.55) & (ALEKS_df['highest_score'] <= 0.64),
    (ALEKS_df['highest_score'] >= 0.65)
]

# Apply the conditions to set the appropriate score_range column to 1
# Non-matching rows will have NaN for these new columns, as requested (no explicit 0 initialization)
for i, cond in enumerate(conditions):
    ALEKS_df.loc[cond, f'score_range_{i}'] = 1

# Drop the temporary 'highest_score' column
ALEKS_df = ALEKS_df.drop(columns=['highest_score'])

ALEKS_df.head()

In [ ]:
hot_all_coursework_df = current_cohort_coursework_df.copy()
hot_all_coursework_df['unique_id'] = hot_all_coursework_df['unique_id'].astype(str).str.zfill(9)
hot_all_coursework_df = hot_all_coursework_df.drop(['datetime_course_created', 'cat_course_external_ref_id', 'num_course_credit', 'cat_course_term_type'], axis=1)

# Initialize the new column 'Course_Sum_work' to 'NONE' by default
hot_all_coursework_df['Course_Sum_work'] = 'NONE'

# Define the mapping rules in a dictionary for easy editing
course_work_math = {
    # MATH
    'ALGEBRA': {'includes': ['ALG'], 'excludes': ['PRE']},
    'GEOMETRY': {'includes': ['GEOM'], 'excludes': []},
    'CALCULUS': {'includes': ['CALC'], 'excludes': ['PRE']},
    'PRECALC': {'includes': ['PRE'], 'excludes': []},
    'STATS': {'includes': ['STAT'], 'excludes': ['CALC']},
    'TRIG': {'includes': ['TRIG'], 'excludes': []},
    'IB MATH SL': {'includes': ['MATH'], 'includes_or': ['IB', 'SL', 'HL'], 'excludes': []},
    'MATH LIBERAL': {'includes': ['MATH'], 'excludes': ['IB', 'STATS', 'SL', 'HL', 'CALC', 'STAT']},
    'DATA ANALYSIS': {'includes': ['DATA', 'ANALYSIS'], 'excludes': []}
}
course_work_sci = {
    # SCI
    'BIOL': {'includes': ['BIOL'], 'excludes': []},
    'CHEM': {'includes': ['CHEM'], 'excludes': []},
    'PHYSICS': {'includes': ['PHYS'], 'excludes': ['PHYSICAL']},
    'PHYSICAL SCIENCE': {'includes': ['PHYSICAL', 'SCIENCE'], 'excludes': ['PHYSICS']},
    'ANATPHYSIO': {'includes': ['ANAT', 'PHYSIO'], 'excludes': []},
    'ANATOMY': {'includes': ['ANAT'], 'excludes': ['PHYS']},
    'BIOMEDICAL SCIENCE': {'includes': ['BIOMED'], 'excludes': ['TERM']},
    'EARTH SCIENCE': {'includes': ['EARTH', 'SCI'], 'excludes': []},
    'FOOD SCIENCE': {'includes': ['FOOD', 'SCI'], 'excludes': []}
}
course_work_mappings = {**course_work_math, **course_work_sci}

# Apply the rules to create 'Course_Sum_work'
for course_type, conditions in course_work_mappings.items():
    current_includes_condition = pd.Series(True, index=hot_all_coursework_df.index)

    # Handle 'includes' (AND conditions)
    if 'includes' in conditions and conditions['includes']:
        for inc_val in conditions['includes']:
            current_includes_condition = current_includes_condition & hot_all_coursework_df['cat_course_name'].str.contains(inc_val, case=False, na=False)

    # Handle 'includes_or' (OR conditions) for inclusions
    if 'includes_or' in conditions and conditions['includes_or']:
        or_part_condition = pd.Series(False, index=hot_all_coursework_df.index)
        for or_inc_val in conditions['includes_or']:
            or_part_condition = or_part_condition | hot_all_coursework_df['cat_course_name'].str.contains(or_inc_val, case=False, na=False)
        current_includes_condition = current_includes_condition & or_part_condition

    # Handle 'excludes' (AND NOT conditions)
    current_excludes_condition = pd.Series(False, index=hot_all_coursework_df.index)
    if 'excludes' in conditions and conditions['excludes']:
        for exc_val in conditions['excludes']:
            current_excludes_condition = current_excludes_condition | hot_all_coursework_df['cat_course_name'].str.contains(exc_val, case=False, na=False)

    hot_all_coursework_df.loc[current_includes_condition & ~current_excludes_condition, 'Course_Sum_work'] = course_type

hot_all_coursework_df = hot_all_coursework_df[hot_all_coursework_df['Course_Sum_work'] != 'NONE']

# New logic for cat_course_lvl_type
# Dictionary for updating cat_course_lvl_type based on conditions
level_type_mappings = {
    'PSEO': {'includes': ['PSEO'], 'includes_or': [], 'excludes': []},
    'AP': {'includes': [], 'includes_or': ['AP', 'ADVANCED PLACEMENT'], 'excludes': []},
    'IB': {'includes': ['IB'], 'includes_or': [], 'excludes': []},
    'HONORS': {'includes': [], 'includes_or': ['HONORS', 'HON'], 'excludes': []},
    'ADVANCED': {'includes': [], 'includes_or': ['ADVANCED', 'ADV'], 'excludes': ['PLACEMENT']},
    'CIS': {'includes': ['CIS'], 'includes_or': [], 'excludes': []},
    'ACCELLERATED': {'includes': [], 'includes_or': ['ACCELERATED', 'ACCEL'], 'excludes': []},
    'PLTW': {'includes': ['PLTW'], 'includes_or': [], 'excludes': []}
}

for level_type, conditions in level_type_mappings.items():
    # Condition 1: cat_course_lvl_type is blank/NaN or an empty string
    blank_level_condition = hot_all_coursework_df['cat_course_lvl_type'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_lvl_type'].isna()

    current_level_includes_condition = pd.Series(True, index=hot_all_coursework_df.index)
    # Handle 'includes' (AND conditions) for cat_course_name
    if 'includes' in conditions and conditions['includes']:
        for inc_val in conditions['includes']:
            current_level_includes_condition = current_level_includes_condition & hot_all_coursework_df['cat_course_name'].str.contains(inc_val, case=False, na=False)

    # Handle 'includes_or' (OR conditions) for cat_course_name
    if 'includes_or' in conditions and conditions['includes_or']:
        or_part_condition = pd.Series(False, index=hot_all_coursework_df.index)
        for or_inc_val in conditions['includes_or']:
            or_part_condition = or_part_condition | hot_all_coursework_df['cat_course_name'].str.contains(or_inc_val, case=False, na=False)
        current_level_includes_condition = current_level_includes_condition & or_part_condition

    current_level_excludes_condition = pd.Series(False, index=hot_all_coursework_df.index)
    # Handle 'excludes' (AND NOT conditions) for cat_course_name
    if 'excludes' in conditions and conditions['excludes']:
        for exc_val in conditions['excludes']:
            current_level_excludes_condition = current_level_excludes_condition | hot_all_coursework_df['cat_course_name'].str.contains(exc_val, case=False, na=False)

    # Apply the update where blank_level_condition AND (current_level_includes_condition AND NOT current_level_excludes_condition) are met
    hot_all_coursework_df.loc[
        blank_level_condition & current_level_includes_condition & ~current_level_excludes_condition,
        'cat_course_lvl_type'
    ] = level_type

# After attempting to categorize, if the field is still empty, set it to "REGULAR"
hot_all_coursework_df.loc[
    hot_all_coursework_df['cat_course_lvl_type'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_lvl_type'].isna(),
    'cat_course_lvl_type'
] = 'REGULAR'

# Condition 1: If 'cat_course_subject' is blank and 'cat_course_math_sci' is 'MATH' or 'SCIENCE'
blank_subject_condition = hot_all_coursework_df['cat_course_subject'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_subject'].isna()
math_sci_condition = hot_all_coursework_df['cat_course_math_sci'].isin(['MATH', 'SCIENCE'])
hot_all_coursework_df.loc[blank_subject_condition & math_sci_condition, 'cat_course_subject'] = hot_all_coursework_df['cat_course_math_sci']

# Condition 2: If 'cat_course_math_sci' is blank, infer from 'Course_Sum_work'
blank_math_sci_condition = hot_all_coursework_df['cat_course_math_sci'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_math_sci'].isna()

# Get lists of Course_Sum_work values for math and science
math_sum_works = list(course_work_math.keys())
sci_sum_works = list(course_work_sci.keys())

# Apply 'MATH' if Course_Sum_work is in math_sum_works
hot_all_coursework_df.loc[
    blank_math_sci_condition & hot_all_coursework_df['Course_Sum_work'].isin(math_sum_works),
    'cat_course_subject'
] = 'MATH'

# Apply 'SCIENCE' if Course_Sum_work is in sci_sum_works (only if not already set to 'MATH')
hot_all_coursework_df.loc[
    blank_math_sci_condition & hot_all_coursework_df['Course_Sum_work'].isin(sci_sum_works) &
    (hot_all_coursework_df['cat_course_subject'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_subject'].isna()),
    'cat_course_subject'
] = 'SCIENCE'

# Drop 'cat_course_name' and 'cat_course_math_sci' as they are processed
hot_all_coursework_df = hot_all_coursework_df.drop(['cat_course_name', 'cat_course_math_sci'], axis=1)

# Now, implement the user's specific request:
# For each unique_id, I want:
#   - a column for each course work type (ALGEBRA - FOOD SCIENCE) with its grade
#   - a column for each course work type (ALGEBRA - FOOD SCIENCE) with its level_type

# Step 1: Prepare the grades DataFrame
# Group by unique_id and Course_Sum_work, then take the mean of num_course_grade
df_grades = hot_all_coursework_df.groupby(['unique_id', 'Course_Sum_work'])['num_course_grade'].mean().unstack()
df_grades = df_grades.add_suffix('_grade').reset_index()

# Step 2: Prepare the level types DataFrame
# Group by unique_id and Course_Sum_work, then aggregate cat_course_lvl_type
# If a student has multiple courses of the same Course_Sum_work but different cat_course_lvl_type,
# we'll represent it as a comma-separated string of unique level types.
def aggregate_level_types(series):
    unique_levels = series.dropna().astype(str).unique()
    if len(unique_levels) == 0:
        return np.nan
    elif len(unique_levels) == 1:
        return unique_levels[0]
    else:
        return ', '.join(sorted(unique_levels)) # Sort for consistent representation

df_level_types = hot_all_coursework_df.groupby(['unique_id', 'Course_Sum_work'])['cat_course_lvl_type'].apply(aggregate_level_types).unstack()
df_level_types = df_level_types.add_suffix('_level_type').reset_index()

# Step 3: Merge the two DataFrames
coursework_flattened_df = pd.merge(df_grades, df_level_types, on='unique_id', how='outer')

# Display the head of the final DataFrame
print("Transformed coursework DataFrame (coursework_flattened_df):")
print(coursework_flattened_df.shape)
coursework_flattened_df.head()

In [ ]:
hot_all_coursework_df = current_cohort_coursework_df.copy()
hot_all_coursework_df['unique_id'] = hot_all_coursework_df['unique_id'].astype(str).str.zfill(9)
hot_all_coursework_df = hot_all_coursework_df.drop(['datetime_course_created', 'cat_course_external_ref_id', 'num_course_credit', 'cat_course_term_type'], axis=1)

# Initialize the new column 'Course_Sum_work' to 'NONE' by default
hot_all_coursework_df['Course_Sum_work'] = 'NONE'

# Define the mapping rules in a dictionary for easy editing
course_work_math = {
    # MATH
    'ALGEBRA': {'includes': ['ALG'], 'excludes': ['PRE']},
    'GEOMETRY': {'includes': ['GEOM'], 'excludes': []},
    'CALCULUS': {'includes': ['CALC'], 'excludes': ['PRE']},
    'PRECALC': {'includes': ['PRE'], 'excludes': []},
    'STATS': {'includes': ['STAT'], 'excludes': ['CALC']},
    'TRIG': {'includes': ['TRIG'], 'excludes': []},
    'IB MATH SL': {'includes': ['MATH'], 'includes_or': ['IB', 'SL', 'HL'], 'excludes': []},
    'MATH LIBERAL': {'includes': ['MATH'], 'excludes': ['IB', 'STATS', 'SL', 'HL', 'CALC', 'STAT']},
    'DATA ANALYSIS': {'includes': ['DATA', 'ANALYSIS'], 'excludes': []}
}
course_work_sci = {
    # SCI
    'BIOL': {'includes': ['BIOL'], 'excludes': []},
    'CHEM': {'includes': ['CHEM'], 'excludes': []},
    'PHYSICS': {'includes': ['PHYS'], 'excludes': ['PHYSICAL']},
    'PHYSICAL SCIENCE': {'includes': ['PHYSICAL', 'SCIENCE'], 'excludes': ['PHYSICS']},
    'ANATPHYSIO': {'includes': ['ANAT', 'PHYSIO'], 'excludes': []},
    'ANATOMY': {'includes': ['ANAT'], 'excludes': ['PHYS']},
    'BIOMEDICAL SCIENCE': {'includes': ['BIOMED'], 'excludes': ['TERM']},
    'EARTH SCIENCE': {'includes': ['EARTH', 'SCI'], 'excludes': []},
    'FOOD SCIENCE': {'includes': ['FOOD', 'SCI'], 'excludes': []}
}
course_work_mappings = {**course_work_math, **course_work_sci}

# Apply the rules to create 'Course_Sum_work'
for course_type, conditions in course_work_mappings.items():
    current_includes_condition = pd.Series(True, index=hot_all_coursework_df.index)

    # Handle 'includes' (AND conditions)
    if 'includes' in conditions and conditions['includes']:
        for inc_val in conditions['includes']:
            current_includes_condition = current_includes_condition & hot_all_coursework_df['cat_course_name'].str.contains(inc_val, case=False, na=False)

    # Handle 'includes_or' (OR conditions) for inclusions
    if 'includes_or' in conditions and conditions['includes_or']:
        or_part_condition = pd.Series(False, index=hot_all_coursework_df.index)
        for or_inc_val in conditions['includes_or']:
            or_part_condition = or_part_condition | hot_all_coursework_df['cat_course_name'].str.contains(or_inc_val, case=False, na=False)
        current_includes_condition = current_includes_condition & or_part_condition

    # Handle 'excludes' (AND NOT conditions)
    current_excludes_condition = pd.Series(False, index=hot_all_coursework_df.index)
    if 'excludes' in conditions and conditions['excludes']:
        for exc_val in conditions['excludes']:
            current_excludes_condition = current_excludes_condition | hot_all_coursework_df['cat_course_name'].str.contains(exc_val, case=False, na=False)

    hot_all_coursework_df.loc[current_includes_condition & ~current_excludes_condition, 'Course_Sum_work'] = course_type

hot_all_coursework_df = hot_all_coursework_df[hot_all_coursework_df['Course_Sum_work'] != 'NONE']

# New logic for cat_course_lvl_type
# Dictionary for updating cat_course_lvl_type based on conditions
level_type_mappings = {
    'PSEO': {'includes': ['PSEO'], 'includes_or': [], 'excludes': []},
    'AP': {'includes': [], 'includes_or': ['AP', 'ADVANCED PLACEMENT'], 'excludes': []},
    'IB': {'includes': ['IB'], 'includes_or': [], 'excludes': []},
    'HONORS': {'includes': [], 'includes_or': ['HONORS', 'HON'], 'excludes': []},
    'ADVANCED': {'includes': [], 'includes_or': ['ADVANCED', 'ADV'], 'excludes': ['PLACEMENT']},
    'CIS': {'includes': ['CIS'], 'includes_or': [], 'excludes': []},
    'ACCELLERATED': {'includes': [], 'includes_or': ['ACCELERATED', 'ACCEL'], 'excludes': []},
    'PLTW': {'includes': ['PLTW'], 'includes_or': [], 'excludes': []}
}

for level_type, conditions in level_type_mappings.items():
    # Condition 1: cat_course_lvl_type is blank/NaN or an empty string
    blank_level_condition = hot_all_coursework_df['cat_course_lvl_type'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_lvl_type'].isna()

    current_level_includes_condition = pd.Series(True, index=hot_all_coursework_df.index)
    # Handle 'includes' (AND conditions) for cat_course_name
    if 'includes' in conditions and conditions['includes']:
        for inc_val in conditions['includes']:
            current_level_includes_condition = current_level_includes_condition & hot_all_coursework_df['cat_course_name'].str.contains(inc_val, case=False, na=False)

    # Handle 'includes_or' (OR conditions) for cat_course_name
    if 'includes_or' in conditions and conditions['includes_or']:
        or_part_condition = pd.Series(False, index=hot_all_coursework_df.index)
        for or_inc_val in conditions['includes_or']:
            or_part_condition = or_part_condition | hot_all_coursework_df['cat_course_name'].str.contains(or_inc_val, case=False, na=False)
        current_level_includes_condition = current_level_includes_condition & or_part_condition

    current_level_excludes_condition = pd.Series(False, index=hot_all_coursework_df.index)
    # Handle 'excludes' (AND NOT conditions) for cat_course_name
    if 'excludes' in conditions and conditions['excludes']:
        for exc_val in conditions['excludes']:
            current_level_excludes_condition = current_level_excludes_condition | hot_all_coursework_df['cat_course_name'].str.contains(exc_val, case=False, na=False)

    # Apply the update where blank_level_condition AND (current_level_includes_condition AND NOT current_level_excludes_condition) are met
    hot_all_coursework_df.loc[
        blank_level_condition & current_level_includes_condition & ~current_level_excludes_condition,
        'cat_course_lvl_type'
    ] = level_type

# Condition 1: If 'cat_course_subject' is blank and 'cat_course_math_sci' is 'MATH' or 'SCIENCE'
blank_subject_condition = hot_all_coursework_df['cat_course_subject'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_subject'].isna()
math_sci_condition = hot_all_coursework_df['cat_course_math_sci'].isin(['MATH', 'SCIENCE'])
hot_all_coursework_df.loc[blank_subject_condition & math_sci_condition, 'cat_course_subject'] = hot_all_coursework_df['cat_course_math_sci']

# Condition 2: If 'cat_course_math_sci' is blank, infer from 'Course_Sum_work'
blank_math_sci_condition = hot_all_coursework_df['cat_course_math_sci'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_math_sci'].isna()

# Get lists of Course_Sum_work values for math and science
math_sum_works = list(course_work_math.keys())
sci_sum_works = list(course_work_sci.keys())

# Apply 'MATH' if Course_Sum_work is in math_sum_works
hot_all_coursework_df.loc[
    blank_math_sci_condition & hot_all_coursework_df['Course_Sum_work'].isin(math_sum_works),
    'cat_course_subject'
] = 'MATH'

# Apply 'SCIENCE' if Course_Sum_work is in sci_sum_works (only if not already set to 'MATH')
hot_all_coursework_df.loc[
    blank_math_sci_condition & hot_all_coursework_df['Course_Sum_work'].isin(sci_sum_works) &
    (hot_all_coursework_df['cat_course_subject'].astype(str).str.strip().eq('') | hot_all_coursework_df['cat_course_subject'].isna()),
    'cat_course_subject'
] = 'SCIENCE'

# Drop 'cat_course_name' and 'cat_course_math_sci' as they are processed
hot_all_coursework_df = hot_all_coursework_df.drop(['cat_course_name', 'cat_course_math_sci'], axis=1)


# NEW LOGIC FOR AGGREGATION:

# Step 0: Removed custom sorting order for cat_course_lvl_type as per new request

# Step 1 & 2: Sort and Deduplicate to select the 'best' course record per unique_id and Course_Sum_work
# Sort by unique_id, Course_Sum_work, then by cat_course_grade_level (descending), then by grade (ascending)
hot_all_coursework_df_filtered = hot_all_coursework_df.sort_values(
    by=['unique_id', 'Course_Sum_work', 'cat_course_grade_level', 'num_course_grade'],
    ascending=[True, True, False, True] # False for grade level (greater), True for num_course_grade (lowest)
)

# Keep the first occurrence for each group, which will be the 'best' based on our sorting
hot_all_coursework_df_filtered = hot_all_coursework_df_filtered.drop_duplicates(
    subset=['unique_id', 'Course_Sum_work'], keep='first'
)

# Step 3: Prepare the grades DataFrame
# Group by unique_id and Course_Sum_work, then take the num_course_grade
df_grades = hot_all_coursework_df_filtered.groupby(['unique_id', 'Course_Sum_work'])['num_course_grade'].first().unstack()
df_grades = df_grades.add_suffix('_grade').reset_index()

# Step 4: Prepare the level types DataFrame
# Group by unique_id and Course_Sum_work, then take the cat_course_lvl_type
df_level_types = hot_all_coursework_df_filtered.groupby(['unique_id', 'Course_Sum_work'])['cat_course_lvl_type'].first().unstack()
df_level_types = df_level_types.add_suffix('_level_type').reset_index()

# Step 5: Merge the two DataFrames
coursework_flattened_df = pd.merge(df_grades, df_level_types, on='unique_id', how='outer')

# Display the head of the final DataFrame
print("Transformed coursework DataFrame (coursework_flattened_df):")
print(coursework_flattened_df.shape)
coursework_flattened_df.head()

## Current Cohort

In [ ]:
current_cohort_apps_df['UMNID'] = current_cohort_apps_df['UMNID'].astype(str).str.zfill(7)
current_cohort_apps_df['unique_id'] = current_cohort_apps_df['unique_id'].astype(str).str.zfill(9)

current_apps_ALEKs_df = pd.merge(current_cohort_apps_df, ALEKS_df,
                                   left_on='UMNID',
                                   right_on='studentInfo_id',
                                   how='left')

# Drop the 'studentInfo_id' column after merging
current_apps_ALEKs_df = current_apps_ALEKs_df.drop(columns=['studentInfo_id'])

current_apps_merged_df = pd.merge(current_apps_ALEKs_df, coursework_flattened_df,
                                   left_on='unique_id',
                                   right_on='unique_id',
                                   how='left')

# - - - TEMP
#current_apps_merged_df = pd.merge(current_cohort_apps_df, ALEKS_df,
#                                   left_on='UMNID',
#                                   right_on='studentInfo_id',
#                                   how='left')

#current_apps_merged_df = current_apps_merged_df.drop(columns=['studentInfo_id'])

# - - - TEMP

# Drop the 'unique_id' column after merging
current_apps_merged_df = current_apps_merged_df.drop(columns=['unique_id'])

# Define the output file name, save df to excel, download xlsx
#output_file_name = 'current_apps_merged.xlsx'
#current_apps_merged_df.to_excel(output_file_name, index=False)
#files.download(output_file_name)

current_apps_merged_df.shape

In [ ]:
# For Current Cohort df
New_Working_df = current_apps_merged_df.copy()
New_Working_Hot_df = New_Working_df.copy()

# Convert specified columns to object (string) type
object_cols = ['HS ZIP', 'High School Code', 'Person Native Pacific Islander Origin',
               'cat_course_grade_level_0', 'cat_course_grade_level_1',
               'cat_course_grade_level_2', 'cat_course_grade_level_3',
               'cat_course_grade_level_4', 'cat_course_grade_level_5',
               'cat_course_grade_level_6', 'cat_course_grade_level_7',
               'cat_course_grade_level_8', 'cat_course_grade_level_9',
               'cat_course_grade_level_10']
# drop proceeding '.0'
for col in object_cols:
    try:
        New_Working_Hot_df[col] = New_Working_Hot_df[col].astype(str).str.rstrip('.0')
        New_Working_Hot_df[col] = New_Working_Hot_df[col].astype(str)
    except:
        pass

# Convert specified columns to boolean type
bool_cols = ['FGEN SG', 'FGEN RC', 'Slate FGEN Person',
             'Slate FGEN App', 'placementResults_isProctored_0']
for col in bool_cols:
  try:
    New_Working_Hot_df[col] = New_Working_Hot_df[col].astype(bool)
  except:
    pass

# Convert specified columns to nullable integer type to handle NaN values
int_cols = ['Class Rank', 'Class Size', 'placementResults_placementIndex_0',
            'placementResults_placementIndex_1', 'placementResults_placementIndex_2',
            'placementResults_placementIndex_3', 'placementResults_placementIndex_4',
            'placementResults_nbPlacementsTaken_0', 'placementResults_nbPlacementsTaken_1',
            'progressInfo_lastLearningProgress_nbTopicsLearned_0',
            'placementResults_isTimedOut_0', 'placementResults_duration_0',
            'placementResults_duration_1', 'placementResults_duration_2',
            'placementResults_duration_3', 'placementResults_duration_4',
            'progressInfo_performance_nbTopicsMastered_0', 'progressInfo_performance_nbTopicsMastered_1',
            'progressInfo_performance_nbTopicsLearned_0', 'progressInfo_performance_nbTopicsLearned_1',
            'progressInfo_performance_totalNbTopics_0', 'studentInfo_totalTime_0',
            'studentInfo_totalTime_1', 'studentInfo_totalTime_2', 'studentInfo_totalTime_3', 'studentInfo_totalTime_4', 'studentInfo_totalTime_5',
            'knowledgeCheckInfo_duration_0', 'knowledgeCheckInfo_duration_1',
            'knowledgeCheckInfo_duration_2', 'knowledgeCheckInfo_duration_3', 'knowledgeCheckInfo_duration_4',
            'knowledgeCheckInfo_performance_nbTopicsMastered_0', 'knowledgeCheckInfo_performance_nbTopicsMastered_1',
            'knowledgeCheckInfo_performance_nbTopicsMastered_2', 'knowledgeCheckInfo_performance_nbTopicsMastered_3', 'knowledgeCheckInfo_performance_nbTopicsMastered_4',
            'knowledgeCheckInfo_performance_totalNbTopics_0', 'progressInfo_lastLearningProgress_duration_0',
            'progressInfo_lastLearningProgress_nbTopicsLearnedPerHour_0']

for col in int_cols:
  try:
    New_Working_Hot_df[col] = New_Working_Hot_df[col].astype('Int64') # Use nullable integer type
  except:
        pass
New_Working_Hot_df.head(3)

## Prior Chort

In [ ]:
past_cohort_help_working_df = past_cohort_help_df.drop(['unique_id'], axis=1)
past_cohort_help_working_df['UMNID'] = past_cohort_help_working_df['UMNID'].astype(str).str.zfill(7)

# Identify UMNIDs in past_past_cohort_apps_df not present in past_cohort_help_working_df
missing_umnids = past_past_cohort_apps_df[~past_past_cohort_apps_df['UMNID'].astype(str).str.zfill(7).isin(past_cohort_help_working_df['UMNID'])]['UMNID'].astype(str).str.zfill(7).unique()

# Create a DataFrame for these missing UMNIDs with HELPMATH and HELPSCI set to 0
new_help_entries = pd.DataFrame({
    'UMNID': missing_umnids,
    'HELPMATH': 0,
    'HELPSCI': 0
})

# Concatenate the new entries to past_cohort_help_working_df
past_cohort_help_working_df = pd.concat([past_cohort_help_working_df, new_help_entries], ignore_index=True)

past_cohort_help_working_df['HELPMATH'] = 0 # TEMP set all help columns to "No" (0) to test dataset without external flags
past_cohort_help_working_df['HELPSCI'] = 0

# Filter UMR_Courses_Hot_df for 'MATH' and 'SCIENCE' type and get relevant IDs
math_course_ids = UMR_Courses_Hot_df[UMR_Courses_Hot_df['Type'] == 'MATH']['ID'].astype(str).unique()
science_course_ids = UMR_Courses_Hot_df[UMR_Courses_Hot_df['Type'] == 'SCIENCE']['ID'].astype(str).unique()

# Identify students in past_cohort_single_help_df whose HELP* is 0 AND have a matching MATH course ID
math_condition_to_update = ((past_cohort_help_working_df['HELPMATH'] == 0) &
                      (past_cohort_help_working_df['UMNID'].isin(math_course_ids)))
sci_condition_to_update = ((past_cohort_help_working_df['HELPSCI'] == 0) &
                      (past_cohort_help_working_df['UMNID'].isin(science_course_ids)))

# Set HELP* to 1 for these students
past_cohort_help_working_df.loc[math_condition_to_update, 'HELPMATH'] = 1
past_cohort_help_working_df.loc[sci_condition_to_update, 'HELPSCI'] = 1
past_cohort_help_working_df.head()

# Math Only
past_cohort_single_help_df = past_cohort_help_working_df.drop(['HELPSCI'], axis=1)
past_cohort_single_help_df = past_cohort_single_help_df.rename(columns={'HELPMATH': 'Help_Y'})
past_cohort_single_help_df.head()

# Sci Only
#past_cohort_single_help_df = past_cohort_help_working_df.drop(['HELPMATH'], axis=1)
#past_cohort_single_help_df = past_cohort_single_help_df.rename(columns={'HELPSCI': 'Help_Y'})
#past_cohort_single_help_df.head()

# Get UMNIDs from past_past_cohort_apps_df
#past_past_UMNIDs = past_past_cohort_apps_df['UMNID'].astype(str).str.zfill(7).unique()

# Filter past_cohort_single_help_df for matching UMNIDs and Help_Y = 1
#filtered_help_df = past_cohort_single_help_df[
#    (past_cohort_single_help_df['UMNID'].isin(past_past_UMNIDs)) &
#    (past_cohort_single_help_df['Help_Y'] == 1)
#]

#display(filtered_help_df)

In [ ]:
past_cohort_apps_df['UMNID'] = past_cohort_apps_df['UMNID'].astype(str).str.zfill(7)
past_cohort_apps_df['unique_id'] = past_cohort_apps_df['unique_id'].astype(str).str.zfill(9)
print(len(past_cohort_apps_df))
past_past_cohort_apps_df['UMNID'] = past_past_cohort_apps_df['UMNID'].astype(str).str.zfill(7)
past_past_cohort_apps_df['unique_id'] = past_past_cohort_apps_df['unique_id'].astype(str).str.zfill(9)
print(len(past_past_cohort_apps_df))

both_past_cohort_apps_df = pd.concat([past_cohort_apps_df, past_past_cohort_apps_df], ignore_index=True)

prior_apps_ALEKs_df = pd.merge(both_past_cohort_apps_df, ALEKS_df,
                                   left_on='UMNID',
                                   right_on='studentInfo_id',
                                   how='left')

# Drop the 'studentInfo_id' column after merging
prior_apps_ALEKs_df = prior_apps_ALEKs_df.drop(columns=['studentInfo_id'])

prior_apps_merged_df = pd.merge(prior_apps_ALEKs_df, coursework_flattened_df,
                                   left_on='unique_id',
                                   right_on='unique_id',
                                   how='left')


prior_apps_merged_df = pd.merge(prior_apps_merged_df, past_cohort_single_help_df,
                                   left_on='UMNID',
                                   right_on='UMNID',
                                   how='left')

# - - - TEMP
#prior_apps_merged_df = pd.merge(past_cohort_apps_df, ALEKS_df,
#                                   left_on='UMNID',
#                                   right_on='studentInfo_id',
#                                   how='left')
#prior_apps_merged_df = prior_apps_merged_df.drop(columns=['studentInfo_id'])

#prior_apps_merged_df = pd.merge(prior_apps_merged_df, past_cohort_single_help_df,
#                                   left_on='UMNID',
#                                   right_on='UMNID',
#                                   how='left')
# - - - TEMP

prior_apps_merged_df['Help_Y'] = prior_apps_merged_df['Help_Y'].fillna(0)

# Drop the 'unique_id' column after merging
prior_apps_merged_df = prior_apps_merged_df.drop(columns=['unique_id','UMNID'])

prior_apps_merged_df.head(3)
print(len(prior_apps_merged_df))

In [ ]:
# For Past Cohort df
Working_df = prior_apps_merged_df.copy()
Working_Hot_df = Working_df.copy()

# Convert specified columns to object (string) type
object_cols = ['HS ZIP', 'High School Code', 'Person Native Pacific Islander Origin',
               'cat_course_grade_level_0', 'cat_course_grade_level_1',
               'cat_course_grade_level_2', 'cat_course_grade_level_3',
               'cat_course_grade_level_4', 'cat_course_grade_level_5',
               'cat_course_grade_level_6', 'cat_course_grade_level_7',
               'cat_course_grade_level_8', 'cat_course_grade_level_9',
               'cat_course_grade_level_10']
# drop proceeding '.0'
for col in object_cols:
    try:
      Working_Hot_df[col] = Working_Hot_df[col].astype(str).str.rstrip('.0')
      Working_Hot_df[col] = Working_Hot_df[col].astype(str)
    except:
        pass

# Convert specified columns to boolean type
bool_cols = ['FGEN SG', 'FGEN RC', 'Slate FGEN Person',
             'Slate FGEN App', 'Help_Y', 'placementResults_isProctored_0']
for col in bool_cols:
    try:
        Working_Hot_df[col] = Working_Hot_df[col].astype(bool)
    except:
        pass

# Convert specified columns to nullable integer type to handle NaN values
int_cols = ['Class Rank', 'Class Size', 'placementResults_placementIndex_0',
            'placementResults_placementIndex_1', 'placementResults_placementIndex_2',
            'placementResults_placementIndex_3', 'placementResults_placementIndex_4',
            'placementResults_nbPlacementsTaken_0', 'placementResults_nbPlacementsTaken_1',
            'progressInfo_lastLearningProgress_nbTopicsLearned_0',
            'placementResults_isTimedOut_0', 'placementResults_duration_0',
            'placementResults_duration_1', 'placementResults_duration_2',
            'placementResults_duration_3', 'placementResults_duration_4',
            'progressInfo_performance_nbTopicsMastered_0', 'progressInfo_performance_nbTopicsMastered_1',
            'progressInfo_performance_nbTopicsLearned_0', 'progressInfo_performance_nbTopicsLearned_1',
            'progressInfo_performance_totalNbTopics_0', 'studentInfo_totalTime_0',
            'studentInfo_totalTime_1', 'studentInfo_totalTime_2', 'studentInfo_totalTime_3', 'studentInfo_totalTime_4', 'studentInfo_totalTime_5',
            'knowledgeCheckInfo_duration_0', 'knowledgeCheckInfo_duration_1',
            'knowledgeCheckInfo_duration_2', 'knowledgeCheckInfo_duration_3', 'knowledgeCheckInfo_duration_4',
            'knowledgeCheckInfo_performance_nbTopicsMastered_0', 'knowledgeCheckInfo_performance_nbTopicsMastered_1',
            'knowledgeCheckInfo_performance_nbTopicsMastered_2', 'knowledgeCheckInfo_performance_nbTopicsMastered_3', 'knowledgeCheckInfo_performance_nbTopicsMastered_4',
            'knowledgeCheckInfo_performance_totalNbTopics_0', 'progressInfo_lastLearningProgress_duration_0',
            'progressInfo_lastLearningProgress_nbTopicsLearnedPerHour_0']

for col in int_cols:
  try:
    Working_Hot_df[col] = Working_Hot_df[col].astype('Int64') # Use nullable integer type
  except:
    pass

Working_Hot_df.head(3)

## OLD

In [ ]:
print("Checking for non-null values in columns unique to current_apps_merged_df:")
found_non_null_columns = []
for col in only_in_current_apps_merged:
    if col in current_apps_merged_df.columns and current_apps_merged_df[col].notna().any():
        found_non_null_columns.append(col)

if found_non_null_columns:
    print("\nYes, the following columns have at least one non-null value:")
    for col in found_non_null_columns:
        print(f"- {col}")
else:
    print("\nNo, all columns unique to current_apps_merged_df are entirely null.")


print(f"\nShape of current_apps_merged_df before correction: {current_apps_merged_df.shape}")

# Identify columns that are entirely null in current_apps_merged_df
all_null_cols_in_current_merged = current_apps_merged_df.columns[current_apps_merged_df.isnull().all()].tolist()

# Identify columns to drop: these must be in 'all_null_cols_in_current_merged' AND in 'only_in_current_apps_merged'
# 'only_in_current_apps_merged' is a list of columns that were in current_apps_merged_df but not in Working_Hot_df
columns_to_drop_based_on_correction = list(set(all_null_cols_in_current_merged) & set(only_in_current_apps_merged))

if columns_to_drop_based_on_correction:
    print(f"Dropping {len(columns_to_drop_based_on_correction)} columns that are entirely null and not in Working_Hot_df.")
    current_apps_merged_df = current_apps_merged_df.drop(columns=columns_to_drop_based_on_correction)
else:
    print("No columns found to drop based on the specified criteria (entirely null AND not in Working_Hot_df).")

print(f"Shape of current_apps_merged_df after correction: {current_apps_merged_df.shape}")

In [ ]:
# Specify the row index you want to inspect
row_index = 210  # You can change this index to view different rows

# Check if the index is within the DataFrame's bounds
if row_index < len(Working_Hot_df):
    print(f"--- Values for row index {row_index} ---")
    for column in Working_Hot_df.columns:
        value = Working_Hot_df.loc[row_index, column]
        print(f"{column}: {value}")
else:
    print(f"Error: Row index {row_index} is out of bounds for the DataFrame.")

In [ ]:
for col, dtype in Working_Hot_df.dtypes.items():
    print(f"{col}: {dtype}")

## Cleaning

In [ ]:
cols_prior = set(prior_apps_merged_df.columns)
cols_current = set(current_apps_merged_df.columns)

# Columns in Working_Hot_df but not in current_apps_merged_df (using cleaned names)
only_in_prior = sorted(list(cols_prior - cols_current))

# Columns in current_apps_merged_df but not in Working_Hot_df
only_in_current = sorted(list(cols_current - cols_prior))

print("Columns in prior df only:")
for col in only_in_prior:
    print(f"- {col}")

print("\nColumns in current df only:")
for col in only_in_current:
    print(f"- {col}")

In [ ]:
print(f"Current cohort shape: {current_apps_merged_df.shape}")
print(f"Prior cohort shape: {prior_apps_merged_df.shape}")


In [ ]:
dummy_application_df = pd.get_dummies(Working_Hot_df)
new_dummy_application_df = pd.get_dummies(New_Working_Hot_df)
dummy_application_df.head(3)

# Option to download as XLSX
#output_xlsx_name = 'dummy_prior.xlsx'
#Working_Hot_df.to_excel(output_xlsx_name, index=False)
#files.download(output_xlsx_name)

In [ ]:
!pip install keras_tuner -q
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# Split our preprocessed data into our features and target arrays

X = dummy_application_df.drop(['Help_Y'], axis=1, errors='ignore')
y = dummy_application_df['Help_Y'].astype(int) # Convert boolean target to integer

# Split the preprocessed data into a training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [ ]:
# Create an imputer that fills missing values with the mean of the column
# Fit only on the training data to avoid data leakage
imputer = SimpleImputer(strategy='mean')

# Create a StandardScaler instances
scaler = StandardScaler()

imputer.fit(X_train)
# Transform both training and testing data
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Fit the StandardScaler
X_scaler = scaler.fit(X_train_imputed)

# Scale the data
X_train_scaled = X_scaler.transform(X_train_imputed)
X_test_scaled = X_scaler.transform(X_test_imputed)

In [ ]:
# Convert X_train_scaled to a DataFrame for easier viewing and export
# Ensure the number of columns provided matches the shape of X_train_scaled
#X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns[:X_train_scaled.shape[1]])

#print("Head of X_train_scaled DataFrame:")
#display(X_train_scaled_df.head())

#output_xlsx_name = 'X_train_scaled.xlsx'
#X_train_scaled_df.to_excel(output_xlsx_name, index=False)

#files.download(output_xlsx_name)

In [ ]:
# 1. Prepare new_dummy_application_df for prediction

new_dummy_application_df = pd.get_dummies(New_Working_Hot_df)
train_features = X.columns

missing_cols_in_new_data = set(train_features) - set(new_dummy_application_df.columns)
for col in missing_cols_in_new_data:
    new_dummy_application_df[col] = 0

# Drop extra columns from new_dummy_application_df that were not in training features
extra_cols_in_new_data = set(new_dummy_application_df.columns) - set(train_features)
new_dummy_application_df = new_dummy_application_df.drop(columns=list(extra_cols_in_new_data))

# Ensure the order of columns matches the training features
New_X_pred = new_dummy_application_df[train_features]

# Impute missing values using the *fitted* imputer (from training data)
New_X_pred_imputed = imputer.transform(New_X_pred)

# Scale the data using the *fitted* scaler (from training data)
New_X_pred_scaled = X_scaler.transform(New_X_pred_imputed)

## Oversample Optimizing

In [ ]:
from imblearn.over_sampling import RandomOverSampler

# Get class counts from y_train to identify minority and majority classes
class_counts = pd.Series(y_train).value_counts()
majority_class_label = class_counts.idxmax()
minority_class_label = class_counts.idxmin()
majority_count = class_counts.max()

# Set the sampling strategy to oversample the minority class to match the majority class count
sampling_strategy = {minority_class_label: majority_count}

# Instantiate the random oversampler model
random_oversampler = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=1)

# Fit and resample the scaled training data
X_train_resampled, y_train_resampled = random_oversampler.fit_resample(X_train_scaled, y_train)

# Update print statements to show shapes before and after resampling
print("X Train Before Resampling: ", X_train_scaled.shape)
print("X Train After Resampling: ", X_train_resampled.shape)

print("Y Train Before Resampling: ", y_train.shape)
print("Y Train After Resampling: ", y_train_resampled.shape)

X_train_scaled = X_train_resampled
y_train = y_train_resampled


## Compile, Train, & Eval Math

In [ ]:
# Define the filepath for saving the weights
math_checkpoint_filepath = 'math_weights_checkpoint.weights.h5'

# Create the ModelCheckpoint callback
math_checkpoint_callback = ModelCheckpoint(
    filepath=math_checkpoint_filepath,
    save_weights_only=True,
    save_freq='epoch'
)

math_early_stopping_callback = EarlyStopping(
    monitor='val_accuracy',
    patience=20,
    mode='max',
    restore_best_weights=True
)

# Define the model
math_model = tf.keras.models.Sequential()

# First hidden layer followed by other
math_model.add(tf.keras.layers.Dense(units=26, activation='relu', input_dim=X_train_scaled.shape[1]))
math_model.add(tf.keras.layers.Dense(units=26, activation='relu'))
math_model.add(tf.keras.layers.Dense(units=1, activation='relu'))
math_model.add(tf.keras.layers.Dense(units=16, activation='relu'))
math_model.add(tf.keras.layers.Dense(units=11, activation='relu'))
math_model.add(tf.keras.layers.Dense(units=21, activation='relu'))

# Output layer
math_model.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

# Check the structure of the model
math_model.summary()

# Compile the model
math_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
import matplotlib.pyplot as plt

history = math_model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test),
                         epochs=100, verbose=1, batch_size=16,
                         callbacks=[math_checkpoint_callback, math_early_stopping_callback])

# Get training history
hist = history.history

# Create a DataFrame for easy plotting if preferred, or directly use the dict
epochs = range(1, len(hist['loss']) + 1)

# Plot training and validation loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, hist['loss'], label='Training Loss')
plt.plot(epochs, hist['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, hist['accuracy'], label='Training Accuracy')
if 'val_accuracy' in hist:
    plt.plot(epochs, hist['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the model using the test data
model_loss, model_accuracy = math_model.evaluate(X_test_scaled,y_test,verbose=1, batch_size=16)
print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")

In [ ]:
## For Math
# 1. Prepare new_dummy_application_df for prediction

missing_cols_in_new_data = set(train_features) - set(new_dummy_application_df.columns)
for col in missing_cols_in_new_data:
    new_dummy_application_df[col] = 0

# 2. Predict Help_Y using the trained model
# Assuming 'math_model' is the final trained model you want to use
predictions_probabilities = math_model.predict(New_X_pred_scaled)

# 3. Add predictions to a copy of New_Working_Hot_df
# Create a copy to avoid modifying the original New_Working_Hot_df directly
New_Working_Hot_df_with_predictions = New_Working_Hot_df.copy()

# Assign the predictions to a new column named 'Predicted_Help_Y'
# Ensure index alignment if New_Working_Hot_df was not reset or modified in a way that altered its index
New_Working_Hot_df_with_predictions['Predicted_Help_Y'] = predictions_probabilities#predictions_binary

# Display the head of the DataFrame with the new prediction column
print("New_Working_Hot_df with 'Predicted_Help_Y' column:")
New_Working_Hot_df_with_predictions.head()

In [ ]:
countupper= New_Working_Hot_df_with_predictions[New_Working_Hot_df_with_predictions['Predicted_Help_Y'] > 0.9].shape[0]
countsatisfactory= New_Working_Hot_df_with_predictions[New_Working_Hot_df_with_predictions['Predicted_Help_Y'] > 0.7].shape[0]
countuncertain= New_Working_Hot_df_with_predictions[New_Working_Hot_df_with_predictions['Predicted_Help_Y'] > 0.4].shape[0]
countlower= New_Working_Hot_df_with_predictions[New_Working_Hot_df_with_predictions['Predicted_Help_Y'] > 0.1].shape[0]

print(f"Number of rows with Predicted_Help_Y > 90% Confidence: {countupper}")
print(f"Number of rows with Predicted_Help_Y > 70% Confidence: {countsatisfactory}")
print(f"Number of rows with Predicted_Help_Y > 40% Confidence: {countuncertain}")
print(f"Number of rows with Predicted_Help_Y > 10% Confidence: {countlower}")

In [ ]:
# For Math
# Define the output file name, save df to excel, download xlsx
output_file_name = 'math_current_apps_help_predictions.xlsx'
New_Working_Hot_df_with_predictions.to_excel(output_file_name, index=False)
files.download(output_file_name)

In [ ]:
# 1. Create a SHAP explainer for the Keras model
# We need a background dataset for the explainer, typically a subset of the training data
# For deep learning models, shap.DeepExplainer is suitable
explainer = shap.DeepExplainer(math_model, X_train_scaled)

# 2. Calculate SHAP values for the test dataset
# This can be computationally intensive for large datasets. Using a smaller sample if needed.
shap_values = explainer.shap_values(X_test_scaled)

shap_values_for_class = None

if isinstance(shap_values, list):
    # For binary classification, usually index 0 represents the negative class and 1 the positive class
    # We are interested in the positive class contribution (Help_Y = 1)
    if len(shap_values) > 1:
        shap_values_for_class = shap_values[0] # Assuming first element is for Help_Y=1
    else:
        # If only one element in list, use that (e.g., for regression or single output)
        temp_shap_values = shap_values[0]
        if temp_shap_values.ndim == 3 and temp_shap_values.shape[-1] == 1:
            shap_values_for_class = temp_shap_values.squeeze(axis=-1);
        else:
            shap_values_for_class = temp_shap_values
elif isinstance(shap_values, np.ndarray):
    # If it's a single numpy array, check its dimensions
    if shap_values.ndim == 3 and shap_values.shape[-1] == 1:
        # Common for single output Keras models: (n_samples, n_features, 1)
        shap_values_for_class = shap_values.squeeze(axis=-1)
    elif shap_values.ndim == 2:
        # Single output, (n_samples, n_features)
        shap_values_for_class = shap_values
    elif shap_values.ndim == 3 and shap_values.shape[0] == 2:
        # Binary classification where shap_values is (2, n_samples, n_features)
        shap_values_for_class = shap_values[0] # Assuming first element is for Help_Y=1
    else:
        raise ValueError(f"Unexpected shape for shap_values ndarray: {shap_values.shape}")
else:
    raise ValueError(f"Unexpected type for shap_values: {type(shap_values)}")

# Ensure shap_values_for_class is now 2D (n_samples, n_features)
if shap_values_for_class is None or shap_values_for_class.ndim != 2:
    raise ValueError("shap_values_for_class could not be correctly processed to a 2D array.")

# 3. Summarize SHAP values to get overall feature importance
# The mean absolute SHAP value for each feature indicates its overall importance
feature_importance = np.abs(shap_values_for_class).mean(axis=0)

# Get feature names from X (training data features)
# Ensure X is the original DataFrame with feature names before one-hot encoding if applicable
feature_names = list(X.columns)

# --- FIX for IndexError: Ensure feature_names length matches feature_importance length ---
# This discrepancy suggests shap.DeepExplainer is returning fewer features than input.
# Truncate feature_names to match the length of feature_importance to avoid IndexError.
if len(feature_names) > len(feature_importance):
    print(f"Warning: Mismatch between number of original features ({len(feature_names)}) and SHAP features ({len(feature_importance)}).")
    print("Truncating feature_names to match SHAP output. Some feature names may be misaligned or missing.")
    feature_names = feature_names[:len(feature_importance)]
elif len(feature_names) < len(feature_importance):
    print(f"Error: SHAP output has more features ({len(feature_importance)}) than original feature names ({len(feature_names)}).")
    raise ValueError("Cannot match SHAP output to fewer original feature names.")
# ----------------------------------------------------------------------------------------

# At this point, feature_importance should be 1-dimensional (n_features,)
# and its length should match len(feature_names).

# Combine feature names and importance into a list of dictionaries for robust DataFrame creation
feature_data = []
for i in range(len(feature_names)):
    feature_data.append({'Feature': feature_names[i], 'Importance': feature_importance[i]})

importance_df = pd.DataFrame(feature_data)

# Calculate correlation between each feature and the target 'y'
# Only consider features that are numerical and exist in X
correlation_data = {}
for feature in importance_df['Feature']:
    if feature in X.columns:
        # Ensure the column is numeric before calculating correlation
        if pd.api.types.is_numeric_dtype(X[feature]):
            correlation_data[feature] = X[feature].corr(y)
        else:
            # For non-numeric features, correlation might not be directly applicable
            correlation_data[feature] = np.nan # Or some other indicator
    else:
        correlation_data[feature] = np.nan # Feature not found in original X

# Add the 'Correlation with Y' column to importance_df
importance_df['Correlation with Y'] = importance_df['Feature'].map(correlation_data)

# Sort by importance in descending order
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# 4. Display the top N features with Importance and Correlation
print("\nTop 20 Features by SHAP Importance (Mean Absolute Value for Math Model) with Correlation:")
display(importance_df.head(20))

## Compile, Train, & Eval Science

In [ ]:
# Define the filepath for saving the weights
sci_checkpoint_filepath = 'sci_weights_checkpoint.weights.h5'

# Create the ModelCheckpoint callback
sci_checkpoint_callback = ModelCheckpoint(
    filepath=sci_checkpoint_filepath,
    save_weights_only=True,
    save_freq='epoch'
)

sci_early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,
    mode='min',
    restore_best_weights=True
)

# Define the model
sci_model = tf.keras.models.Sequential()

# First hidden layer followed by other
sci_model.add(tf.keras.layers.Dense(units=400, activation='relu', input_dim=X_train_scaled.shape[1]))
sci_model.add(tf.keras.layers.Dense(units=400, activation='relu'))
sci_model.add(tf.keras.layers.Dense(units=400, activation='relu'))
sci_model.add(tf.keras.layers.Dense(units=400, activation='relu'))
sci_model.add(tf.keras.layers.Dense(units=400, activation='relu'))
sci_model.add(tf.keras.layers.Dense(units=400, activation='relu'))
sci_model.add(tf.keras.layers.Dense(units=400, activation='relu'))
sci_model.add(tf.keras.layers.Dense(units=500, activation='relu'))


# Output layer
sci_model.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

# Check the structure of the model
sci_model.summary()

# Compile the model
sci_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Train the model
history = sci_model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test), epochs=100, verbose=1, batch_size=16, callbacks=[sci_checkpoint_callback, sci_early_stopping_callback])

# Get training history
hist = history.history

# Create a DataFrame for easy plotting if preferred, or directly use the dict
epochs = range(1, len(hist['loss']) + 1)

# Plot training and validation loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, hist['loss'], label='Training Loss')
plt.plot(epochs, hist['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, hist['accuracy'], label='Training Accuracy')
if 'val_accuracy' in hist:
    plt.plot(epochs, hist['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the model using the test data
model_loss, model_accuracy = sci_model.evaluate(X_test_scaled,y_test,verbose=1, batch_size=16)
print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")

# Export our model to keras file
sci_model.save('Sci_Model_Attempt.keras')

In [ ]:
## For Science
# 1. Prepare new_dummy_application_df for prediction

missing_cols_in_new_data = set(train_features) - set(new_dummy_application_df.columns)
for col in missing_cols_in_new_data:
    new_dummy_application_df[col] = 0

# 2. Predict Help_Y using the trained model
# Assuming 'sci_model' is the final trained model you want to use
predictions_probabilities = sci_model.predict(New_X_pred_scaled)

# 3. Add predictions to a copy of New_Working_Hot_df
# Create a copy to avoid modifying the original New_Working_Hot_df directly
New_Working_Hot_df_with_predictions = New_Working_Hot_df.copy()

# Assign the predictions to a new column named 'Predicted_Help_Y'
# Ensure index alignment if New_Working_Hot_df was not reset or modified in a way that altered its index
New_Working_Hot_df_with_predictions['Predicted_Help_Y'] = predictions_probabilities#predictions_binary

# Display the head of the DataFrame with the new prediction column
print("New_Working_Hot_df with 'Predicted_Help_Y' column:")
New_Working_Hot_df_with_predictions.head()

In [ ]:
# For Science
# Define the output file name, save df to excel, download xlsx
output_file_name = 'sci_current_apps_help_predictions.xlsx'
New_Working_Hot_df_with_predictions.to_excel(output_file_name, index=False)
files.download(output_file_name)

In [ ]:
# Assuming sci_model, X_train_scaled, and X_test_scaled are available from previous cells
# Also assuming X (the original feature DataFrame) is available for column names

# 1. Create a SHAP explainer for the Keras model
# We need a background dataset for the explainer, typically a subset of the training data
# For deep learning models, shap.DeepExplainer is suitable
explainer = shap.DeepExplainer(sci_model, X_train_scaled)

# 2. Calculate SHAP values for the test dataset
# This can be computationally intensive for large datasets. Using a smaller sample if needed.
shap_values = explainer.shap_values(X_test_scaled)

# DeepExplainer can return different structures:
# - A list of arrays (e.g., [shap_values_for_class0, shap_values_for_class1] for binary output)
# - A single 3D array (e.g., (n_samples, n_features, 1) for single output Keras models)
# - A single 2D array (e.g., (n_samples, n_features) for regression or single output class)

shap_values_for_class = None

if isinstance(shap_values, list):
    # For binary classification, usually index 1 represents the positive class
    if len(shap_values) > 1:
        shap_values_for_class = shap_values[1]
    else:
        # If only one element in list, use that (e.g., for regression or single output)
        temp_shap_values = shap_values[0]
        if temp_shap_values.ndim == 3 and temp_shap_values.shape[-1] == 1:
            shap_values_for_class = temp_shap_values.squeeze(axis=-1)
        else:
            shap_values_for_class = temp_shap_values
elif isinstance(shap_values, np.ndarray):
    # If it's a single numpy array, check its dimensions
    if shap_values.ndim == 3 and shap_values.shape[-1] == 1:
        # Common for single output Keras models: (n_samples, n_features, 1)
        shap_values_for_class = shap_values.squeeze(axis=-1)
    elif shap_values.ndim == 2:
        # Single output, (n_samples, n_features)
        shap_values_for_class = shap_values
    elif shap_values.ndim == 3 and shap_values.shape[0] == 2:
        # Binary classification where shap_values is (2, n_samples, n_features)
        shap_values_for_class = shap_values[1]
    else:
        raise ValueError(f"Unexpected shape for shap_values ndarray: {shap_values.shape}")
else:
    raise ValueError(f"Unexpected type for shap_values: {type(shap_values)}")

# Ensure shap_values_for_class is now 2D (n_samples, n_features)
if shap_values_for_class is None or shap_values_for_class.ndim != 2:
    raise ValueError("shap_values_for_class could not be correctly processed to a 2D array.")

# 3. Summarize SHAP values to get overall feature importance
# The mean absolute SHAP value for each feature indicates its overall importance
feature_importance = np.abs(shap_values_for_class).mean(axis=0)

# Get feature names from X (training data features)
# Ensure X is the original DataFrame with feature names before one-hot encoding if applicable
# Explicitly convert to list to avoid potential pandas Index issues with DataFrame constructor
feature_names = list(X.columns)

# --- FIX for IndexError: Ensure feature_names length matches feature_importance length ---
# This discrepancy suggests shap.DeepExplainer is returning fewer features than input.
# Truncate feature_names to match the length of feature_importance to avoid IndexError.
if len(feature_names) > len(feature_importance):
    print(f"Warning: Mismatch between number of original features ({len(feature_names)}) and SHAP features ({len(feature_importance)}).")
    print("Truncating feature_names to match SHAP output. Some feature names may be misaligned or missing.")
    feature_names = feature_names[:len(feature_importance)]
elif len(feature_names) < len(feature_importance):
    print(f"Error: SHAP output has more features ({len(feature_importance)}) than original feature names ({len(feature_names)}).")
    raise ValueError("Cannot match SHAP output to fewer original feature names.")
# ----------------------------------------------------------------------------------------

# At this point, feature_importance should be 1-dimensional (n_features,)
# and its length should match len(feature_names).

# Combine feature names and importance into a list of dictionaries for robust DataFrame creation
feature_data = []
for i in range(len(feature_names)):
    feature_data.append({'Feature': feature_names[i], 'Importance': feature_importance[i]})

importance_df = pd.DataFrame(feature_data)

# Sort by importance in descending order
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# 4. Display the top N features
print("\nTop 20 Features by SHAP Importance (Mean Absolute Value):")
print(importance_df.head(20))

# Optional: SHAP summary plot for a visual overview
# shap.summary_plot(shap_values_for_class, X_test_scaled, feature_names=feature_names, plot_type="bar")
# shap.summary_plot(shap_values_for_class, X_test_scaled, feature_names=feature_names)


## Hyperparameter Tuning with Keras Tuner

In [ ]:
# Function to create a new model for Keras Tuner
def build_model(hp):
    optimize_math_model = tf.keras.models.Sequential()

    # Allow kerastuner to decide which activation function to use in hidden layers
    activation = hp.Choice('activation', ['relu'])

    # Allow kerastuner to decide number of neurons in first layer
    optimize_math_model.add(tf.keras.layers.Dense(units=hp.Int('first_units', min_value=1, max_value=30, step=5),
                                       activation=activation, input_dim=X_train_scaled.shape[1]))

    # Allow kerastuner to decide number of hidden layers and neurons in each layer
    for i in range(hp.Int('num_hidden_layers', 4, 7)):
        optimize_math_model.add(tf.keras.layers.Dense(units=hp.Int('units_' + str(i), min_value=1, max_value=30, step=5),
                                           activation=activation))

    # Output layer: Allow tuner to choose between sigmoid and softmax
    output_activation = hp.Choice('output_activation', ['sigmoid'])
    optimize_math_model.add(tf.keras.layers.Dense(units=1, activation=output_activation))

    # Compile the model
    optimize_math_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    return optimize_math_model

# Define the early stopping callback
early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=7,
    mode='min',
    restore_best_weights=True
)

# Create a RandomSearch tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=100,
    overwrite=True)

# Perform hyperparameter search
tuner.search(X_train_scaled, y_train, epochs=100, validation_data=(X_test_scaled, y_test), callbacks=[early_stopping_callback])

In [ ]:
# Get the best models from the tuner
best_hps = tuner.get_best_hyperparameters(num_trials=1)

# Build the best model
best_model = tuner.hypermodel.build(best_hps[0])

# Display the best hyperparameters
print("Best Hyperparameters found by Keras Tuner:")
for param, value in best_hps[0].values.items():
    print(f"  {param}: {value}")

# Summarize the best model
best_model.summary()

## Building and Training the Optimized Model

In [ ]:
# Retrieve the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the optimized model using the best hyperparameters
nn_model_optimized = tuner.hypermodel.build(best_hps)

# Display the summary of the optimized model
nn_model_optimized.summary()

# Compile the optimized model (already done in build_model, but explicitly here for clarity)
nn_model_optimized.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Train the optimized model
history_optimized = nn_model_optimized.fit(X_train_scaled, y_train, epochs=30, verbose=2, batch_size=16, callbacks=[checkpoint_callback])

In [ ]:
# Evaluate the optimized model using the test data
model_loss_optimized, model_accuracy_optimized = nn_model_optimized.evaluate(X_test_scaled, y_test, verbose=1, batch_size=16)
print(f"Optimized Model Loss: {model_loss_optimized}, Optimized Model Accuracy: {model_accuracy_optimized}")

# Export our optimized model to an keras file
nn_model_optimized.save('Optimized_Model.keras')